# Five tokamak measurements into PyTorch

Run this notebook on IBM LinuxONE, with all files from this bundle in one folder.
Press **Shift + Enter** on each code cell, from top to bottom.

This notebook extends beyond the 39-signal TokaMark subset to **FAIR MAST level2**.
All five inputs come from the same real, historical MAST pulse and are fetched
into the machine running Python's RAM. Measurement files are not saved to disk.

Inputs, in order: **plasma current (MA), core electron temperature (keV), derived
n=1 saddle-field amplitude estimate (T), line-average electron density (10^20 m^-3),
total radiated power (MW)**. The magnetic estimate is calculated from toroidally
distributed coils and archived geometry; it includes external-field pickup.
It is not a validated plasma-only instability diagnostic or n1rms measurement.
The geometry has development status. Review it with a fusion specialist.

The final exercise trains a small **future-current forecasting** model. A
disruption predictor additionally needs a defined event target and event labels.


In [ ]:
# This is a Jupyter magic command, not a normal Python statement.
# Install the remote-data reader dependencies into this notebook's Python environment.
# Use IBM's existing PyTorch kernel; this file does not install/replace PyTorch.
%pip install -r requirements-five-features.txt


Use the IBM-provided PyTorch Python kernel. The installation cell adds data
reader libraries; it does not install or replace PyTorch. Plain CPU tensors do
not demonstrate Telum acceleration.


In [ ]:
# platform identifies which Python version and CPU architecture run this notebook.
import platform
# NumPy holds the measurement arrays and performs numerical preprocessing.
import numpy as np
# PyTorch supplies tensors, the neural network, and its training operations.
import torch
# Import our five-feature reader, its normalizer, feature descriptions, and
# iterator that pairs input windows with actual future-current measurements.
from mast_five_features import (
    MASTFiveFeatureStreamer, SensorNormalizer, FEATURE_NAMES, FEATURE_UNITS,
    DEFAULT_FEATURE_MINS, DEFAULT_FEATURE_MAXS, iter_five_feature_pairs,
)
# This output confirms whether execution is on your IBM server or another machine.
print("Python:", platform.python_version(), "Architecture:", platform.machine())
# Record the PyTorch build in use, which matters when reproducing a run.
print("PyTorch:", torch.__version__)
# Each position in this tuple corresponds to one of the tensor's five columns.
print("Feature order:", FEATURE_NAMES)
# These are the physical units before normalization changes the numerical scale.
print("Units:", FEATURE_UNITS)


## Read all five features from one pulse

The grid has 10 steps spaced by 2 ms: a nominal 20 ms window. Each step uses
only a measurement at or before its timestamp. Slow diagnostics repeat their
last finite measurement, up to a documented age limit; no future values are
used. This does not create independent 2 ms temperature measurements.

The loader refuses a missing, stale, or incorrectly unit-labeled diagnostic.
The n=1 fit needs at least six finite coils and a well-conditioned geometry.


In [ ]:
# A shot is one experimental plasma pulse. All five inputs use this same shot.
# The with block automatically closes the remote store when the block ends.
with MASTFiveFeatureStreamer(shot=30421, start_s=0.1) as stream:
    # Fetch one aligned window beginning at 0.100 seconds of experimental time.
    window = stream.read_window()
    # Record exactly which archive location supplied these measurements.
    print("Source:", window.source_url)
    # Axes mean [batch=1, time_steps=10, features=5].
    print("Raw shape:", window.raw_batch.shape)
    # Display the first time step with names attached to its five measurements.
    print("First row:", dict(zip(FEATURE_NAMES, window.values[0].tolist())))
    # Slow diagnostics may repeat a prior reading: show the oldest age per feature.
    # Multiplying by 1000 converts seconds to milliseconds.
    print("Maximum sample ages (ms):", 1000 * window.sample_age_s.max(axis=0))
    # Missing coils are allowed only if enough remain for a stable harmonic fit.
    print("Minimum coils in n=1 fit:", window.minimum_magnetic_coils)
    # A larger residual means the offset+n=1 fit explains less of the coil data.
    print("Magnetic fit RMS residual (T):", window.magnetic_fit_rms_T)
    # Geometry currently has development status; inspect this source limitation.
    print("Geometry status:", stream.geometry_status)


## Convert a window into a PyTorch tensor

These example ranges mirror the proposed normalizer. They are **not universal
physical limits**. Clipping is disabled so out-of-range measurements remain
visible. The later training exercise fits its own bounds on training data only.


In [ ]:
# These five example bounds demonstrate the API; they are not universal limits.
# clip defaults to False, so values beyond the supplied bounds remain visible.
example_normalizer = SensorNormalizer(DEFAULT_FEATURE_MINS, DEFAULT_FEATURE_MAXS)
# Add the batch axis and apply scaling to create a PyTorch float32 CPU tensor.
x = window.as_tensor(example_normalizer)
# This is the input format a compatible model must accept: [1,10,5].
print("Tensor:", x.shape, x.dtype, x.device)
# Every element must be finite before it is sent through a neural network.
print("All finite:", torch.isfinite(x).all().item())


## Select separate training, validation, and test pulses

Reuse the pinned TokaMark shot assignments to keep experiments disjoint.
This expanded FAIR MAST task is not an official TokaMark benchmark score.
Some shots lack a required diagnostic and are explicitly skipped. Increase
the candidate count if no complete examples remain in a split.


In [ ]:
# This small helper reads official shot assignments, not the old one-feature model.
from fusion_data_utils import official_splits

# Start small. Later use 20, 100, or None (all assigned pulses) for more candidates.
LIMIT_PER_SPLIT = 2
# Cap the complete windows collected from each pulse; unavailable windows are skipped.
EXAMPLES_PER_SHOT = 8
# An epoch is one full pass through the collected training examples.
EPOCHS = 3
# Return three separate lists in the order training, validation, test.
assigned = official_splits(strategy="random", limit_per_split=LIMIT_PER_SPLIT)
# Give those lists names so later cells can request the intended split clearly.
splits = dict(zip(("train", "val", "test"), assigned))
# Display candidate pulse IDs before fetching their diagnostics.
for name, shots in splits.items():
    print(name, shots)
# & means set intersection. Each assertion requires no pulse IDs to be shared.
# Keeping an entire pulse in one split prevents experimental information leakage.
assert not (set(splits["train"]) & set(splits["val"]))
assert not (set(splits["train"]) & set(splits["test"]))
assert not (set(splits["val"]) & set(splits["test"]))


## Make small training batches in RAM

Each input is [10,5]. The target is an actual current measurement near 10 ms
after the last input; source sampling may place it up to the current age limit
earlier than the requested target time. It always occurs after the last input.

Only this explicitly capped demonstration subset is collected in RAM. The
underlying archive remains remote. No measurement files are written. Keeping
inputs for a few training epochs in RAM avoids repeatedly fetching the subset.


In [ ]:
# A helper avoids repeating the same fetching/stacking logic for every split.
def collect(shots):
    # Fetch only this small configured subset into RAM. Each yielded pair is
    # x=[10,5] physical measurements and y=[1] actual later plasma current in MA.
    # Missing diagnostics are reported as warnings when skipping is enabled.
    pairs = list(iter_five_feature_pairs(
        shots, start_s=0.1, stop_s=0.5,
        examples_per_shot=EXAMPLES_PER_SHOT, skip_bad_shots=True,
    ))
    # Stack separate examples along a new axis: [examples,10,5] and [examples,1].
    return np.stack([a for a, _ in pairs]), np.stack([b for _, b in pairs])

# Fetch inputs and labels for training. These examples update model weights.
train_raw, train_y = collect(splits["train"])
# Fetch separate validation examples. These evaluate progress without weight updates.
val_raw, val_y = collect(splits["val"])
# Check actual counts: missing diagnostics may reduce the requested subset size.
print("Train:", train_raw.shape, train_y.shape)
print("Validation:", val_raw.shape, val_y.shape)

# Reduce over example and time axes, leaving one minimum for each of five features.
# Only training data may determine the preprocessing applied to all later inputs.
mins = train_raw.min(axis=(0, 1))
# Obtain the corresponding five maxima using the same training-only measurements.
maxs = train_raw.max(axis=(0, 1))
# Give a constant training feature a small positive range to prevent division by zero.
maxs = np.maximum(maxs, mins + np.maximum(np.abs(mins) * 1e-6, 1e-6))
# One fitted normalizer is reused unchanged for validation, test, and later inference.
normalizer = SensorNormalizer(mins, maxs)
# Convert every training input to a normalized float32 CPU tensor.
train_x = normalizer.process_raw_batch(train_raw)
# Apply the TRAINING bounds to validation inputs; do not fit a new normalizer here.
val_x = normalizer.process_raw_batch(val_raw)
# Targets remain physical current in MA, so squared-error units will be MA^2.
train_targets = torch.from_numpy(train_y)
# These are the validation labels, not inputs used to make a prediction.
val_targets = torch.from_numpy(val_y)

# TensorDataset pairs each input with its target; DataLoader groups them into batches.
from torch.utils.data import DataLoader, TensorDataset
# Train with at most eight examples per batch and a reproducible shuffled order.
loader = DataLoader(TensorDataset(train_x, train_targets), batch_size=8,
                    shuffle=True, generator=torch.Generator().manual_seed(42))
# Inspect one batch. This does not update the model or remove stored examples.
batch_x, batch_y = next(iter(loader))
# Expected dimensions: inputs [batch,10,5], targets [batch,1].
print("PyTorch batch:", batch_x.shape, batch_y.shape)


## Train a five-input forecasting model

The network accepts all five features. Loss measures squared error in MA^2.
A falling training loss alone does not show useful predictive performance;
compare held-out errors against simply repeating the latest measured current.


In [ ]:
# nn contains PyTorch's neural-network layers and loss functions.
from torch import nn
# Reproduce the same initial random weights when rerunning this cell.
torch.manual_seed(42)
# Flatten changes [batch,10,5] into [batch,50]. Linear(50,32) mixes the inputs;
# ReLU adds a nonlinear response; Linear(32,1) predicts one future current in MA.
# CPU execution is explicit and does not automatically use the Telum accelerator.
model = nn.Sequential(nn.Flatten(), nn.Linear(10 * 5, 32), nn.ReLU(), nn.Linear(32, 1)).to("cpu")
# Adam adjusts weights using their calculated gradients; lr is the step-size setting.
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# Mean squared error compares predicted current with the actual future current.
criterion = nn.MSELoss()

# Repeat the training pass the configured number of times.
for epoch in range(EPOCHS):
    # Select training behavior for the model's layers.
    model.train()
    # Reset counters used to calculate the epoch's average training error.
    error_sum = 0.0
    examples = 0
    # Each batch contains input tensors and their matching future-current labels.
    for inputs, targets in loader:
        # Discard gradients from the previous batch before calculating new ones.
        optimizer.zero_grad(set_to_none=True)
        # Forward pass: make predictions, then calculate their error against labels.
        loss = criterion(model(inputs), targets)
        # Stop if the training calculation has produced NaN or infinity.
        if not torch.isfinite(loss):
            raise ValueError("Nonfinite training loss")
        # Backpropagation calculates how each weight affects this batch's error.
        loss.backward()
        # THIS is the operation that changes model weights: the learning step.
        optimizer.step()
        # Weight each batch's mean error by its actual number of examples.
        error_sum += loss.item() * len(targets)
        # Count examples so the last, possibly smaller batch is handled correctly.
        examples += len(targets)
    # Switch to evaluation behavior before checking a separate set of pulses.
    model.eval()
    # Validation creates no gradient graph and performs no optimizer updates.
    with torch.inference_mode():
        # Measure error on examples the optimizer did not train on.
        val_mse = criterion(model(val_x), val_targets).item()
    # Baseline: predict that future current equals the last measured current.
    # [:,-1,0:1] selects every example's last time step and plasma-current column.
    persistence_mse = float(np.mean((val_raw[:, -1, 0:1] - val_y) ** 2))
    # Smaller errors are better. Training error alone does not establish usefulness.
    print(f"Epoch {epoch+1}: train={error_sum/examples:.6g}, "
          f"val={val_mse:.6g}, persistence={persistence_mse:.6g} MA^2")


## Evaluate on held-out test pulses once

Use validation to develop the model, and test to measure final performance.
The small demonstration is not a validated scientific predictor.


In [ ]:
# Fetch held-out test pulses only after training and development decisions.
test_raw, test_y = collect(splits["test"])
# Apply the exact normalizer fitted earlier using TRAINING data only.
test_x = normalizer.process_raw_batch(test_raw)
# Use evaluation behavior; these test examples must not train the network.
model.eval()
# Disable gradient tracking while predicting the held-out examples.
with torch.inference_mode():
    # Compare test predictions against real future-current measurements.
    test_mse = criterion(model(test_x), torch.from_numpy(test_y)).item()
# Compare with the same simple last-current persistence baseline used for validation.
baseline_mse = float(np.mean((test_raw[:, -1, 0:1] - test_y) ** 2))
# Report actual test-set size and both squared errors in physical MA^2 units.
print("Test examples:", len(test_y))
print("Test MSE:", test_mse, "MA^2")
print("Persistence MSE:", baseline_mse, "MA^2")


## Connect to your team's model

`raw = stream.fetch_next_20ms_window()` returns a NumPy array [1,10,5].
`inputs = normalizer.process_raw_batch(raw)` returns a float32 CPU tensor with
the same shape. Your model must accept that shape and the documented feature
definitions. More training requires additional complete pulses and a defined
target, not just tensor conversion.

Save the training bounds, feature names, units, derivation, and shot lists with
any model checkpoint so inference uses the same preprocessing. This notebook
does not automatically write measurements or checkpoints to disk.
